# Turkish Legal Dense Retrieval Baseline

## Purpose

This notebook implements the baseline dense retrieval component for the Turkish Legal RAG pipeline.

**Architecture:**
```
Question → Embedding → Vector Search → Results
```

**What this notebook does:**
- Loads the retrieval corpus prepared in notebook 02
- Encodes all corpus chunks using a multilingual sentence transformer
- Builds a FAISS vector index for fast similarity search
- Implements a retrieval function for dense search
- Tests retrieval on Turkish legal queries
- Saves embeddings, index, and retrieval results

**What this notebook does NOT do:**
- Implement BM25 or keyword-based retrieval
- Rerank search results
- Generate answers using an LLM
- Evaluate retrieval performance formally
- Use LangChain, LlamaIndex, or other frameworks

**Model:** `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (supports Turkish and 100+ languages)

## 1. Environment Setup

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple
import time

## 2. Google Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("✓ Google Drive mounted")

## 3. Configuration and Paths

In [ ]:
# Project configuration
# ===== KAGGLE-ONLY BASELINE =====
PROJECT_ROOT = "/content/drive/My Drive/nlp-rag-project"
RETRIEVAL_DATA_PATH = f"{PROJECT_ROOT}/data/retrieval/kaggle_retrieval_corpus.csv"
INDEX_OUTPUT_DIR = f"{PROJECT_ROOT}/outputs/dense_retrieval"

# Embedding and retrieval configuration
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
TOP_K = 5
BATCH_SIZE = 64
EMBEDDING_DIMENSION = 384  # MiniLM-L12-v2 outputs 384-dim embeddings

# Create output directory
Path(INDEX_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Kaggle retrieval corpus: {RETRIEVAL_DATA_PATH}")
print(f"Output directory: {INDEX_OUTPUT_DIR}")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Embedding dimension: {EMBEDDING_DIMENSION}")
print(f"Top K: {TOP_K}, Batch size: {BATCH_SIZE}")

## 4. Install and Import Dependencies

In [ ]:
# Install required packages
import subprocess
import sys

print("Installing dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"])
print("✓ Dependencies installed")

In [ ]:
# Import after installation
from sentence_transformers import SentenceTransformer
import faiss

print("✓ Imports successful")
print(f"  - sentence-transformers: {SentenceTransformer.__module__}")
print(f"  - faiss: {faiss.__version__}")

## 5. Load Retrieval Corpus

In [ ]:
# Load retrieval corpus
print(f"Loading retrieval corpus from {RETRIEVAL_DATA_PATH}...")
df_corpus = pd.read_csv(RETRIEVAL_DATA_PATH)

print(f"✓ Loaded retrieval corpus")
print(f"  - Shape: {df_corpus.shape}")
print(f"  - Columns: {list(df_corpus.columns)}")

## 6. Inspect Corpus Structure

In [ ]:
print("="*80)
print("CORPUS INSPECTION")
print("="*80)

print(f"\nCorpus shape: {df_corpus.shape}")
print(f"\nMissing values in key columns:")
print(df_corpus[['chunk_text', 'source', 'category']].isnull().sum())

print(f"\nChunk text length statistics:")
print(df_corpus['chunk_text'].str.len().describe())

In [ ]:
# Check for empty chunk_text
empty_chunks = df_corpus['chunk_text'].isna().sum() + (df_corpus['chunk_text'].str.len() == 0).sum()
print(f"\nRows with empty/null chunk_text: {empty_chunks}")

if empty_chunks > 0:
    print("Dropping empty chunks...")
    df_corpus = df_corpus[df_corpus['chunk_text'].notna()]
    df_corpus = df_corpus[df_corpus['chunk_text'].str.len() > 0]
    print(f"✓ Corpus size after filtering: {len(df_corpus)}")

print(f"\nSample chunk:")
print(df_corpus['chunk_text'].iloc[0][:200])

## 7. Load Embedding Model

In [ ]:
print(f"Loading embedding model: {EMBEDDING_MODEL_NAME}")
print("This may take a few moments...")

model = SentenceTransformer(EMBEDDING_MODEL_NAME)
model.eval()  # Set to evaluation mode

print(f"✓ Model loaded")
print(f"  - Model type: {type(model).__name__}")
print(f"  - Token limit: {model.max_seq_length}")
print(f"  - Output dimension: {model.get_sentence_embedding_dimension()}")

## 8. Generate Corpus Embeddings

In [ ]:
# Prepare texts for embedding
chunk_texts = df_corpus['chunk_text'].tolist()
print(f"Encoding {len(chunk_texts)} chunks...")

# Encode in batches
embeddings = model.encode(
    chunk_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True  # L2 normalize for cosine similarity
)

embeddings = np.array(embeddings, dtype=np.float32)
print(f"✓ Embeddings generated")
print(f"  - Shape: {embeddings.shape}")
print(f"  - Data type: {embeddings.dtype}")
print(f"  - Dimension: {embeddings.shape[1]}")

In [ ]:
# Verify embeddings are normalized
embedding_norms = np.linalg.norm(embeddings, axis=1)
print(f"\nEmbedding norm verification (should be ~1.0):")
print(f"  - Min: {embedding_norms.min():.6f}")
print(f"  - Max: {embedding_norms.max():.6f}")
print(f"  - Mean: {embedding_norms.mean():.6f}")
print(f"  - Std: {embedding_norms.std():.6f}")

## 9. Build FAISS Vector Index

In [ ]:
print(f"Building FAISS index...")

# Create index for cosine similarity search on normalized embeddings
# Using inner product on normalized embeddings = cosine similarity
index = faiss.IndexFlatIP(EMBEDDING_DIMENSION)

# Add embeddings to index
index.add(embeddings)

print(f"✓ FAISS index built")
print(f"  - Index type: {type(index).__name__}")
print(f"  - Number of vectors: {index.ntotal}")
print(f"  - Vector dimension: {index.d}")
print(f"  - Index capacity: {index.ntotal}")

## 10. Implement Retrieval Function

In [ ]:
def retrieve_dense(
    query: str,
    model: SentenceTransformer,
    index: faiss.Index,
    corpus_df: pd.DataFrame,
    top_k: int = 5
) -> pd.DataFrame:
    """
    Retrieve top-k chunks for a query using dense retrieval.
    
    Args:
        query: Query text in Turkish or other supported language
        model: SentenceTransformer model for encoding
        index: FAISS index for similarity search
        corpus_df: DataFrame with corpus metadata
        top_k: Number of results to return
    
    Returns:
        DataFrame with retrieved chunks, sorted by descending similarity score
    """
    # Encode query
    query_embedding = model.encode(
        query,
        normalize_embeddings=True
    )
    query_embedding = np.array([query_embedding], dtype=np.float32)
    
    # Search index
    scores, indices = index.search(query_embedding, top_k)
    
    # Extract results
    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), 1):
        row = corpus_df.iloc[idx].to_dict()
        row['rank'] = rank
        row['similarity_score'] = float(score)
        results.append(row)
    
    # Create result DataFrame
    results_df = pd.DataFrame(results)
    
    # Reorder columns for clarity
    column_order = [
        'rank', 'similarity_score',
        'chunk_id', 'doc_id',
        'source', 'category',
        'question', 'answer',
        'chunk_text'
    ]
    return_cols = [c for c in column_order if c in results_df.columns]
    return results_df[return_cols]


print("✓ Retrieval function defined")

## 11. Test Queries

In [ ]:
# Define Turkish legal test queries
test_queries = [
    "Haksız zenginleşme ile ilgili hükümler nelerdir?",
    "Miras bırakanın tasarruf özgürlüğü nasıl sınırlandırılır?",
    "Ceza muhakemesinde tutuklama şartları nelerdir?",
    "Anayasa'ya göre devletin şekli nedir?",
    "Aile yurdu ile ilgili malik üzerindeki sınırlamalar nelerdir?"
]

print(f"Defined {len(test_queries)} test queries")
for i, q in enumerate(test_queries, 1):
    print(f"  {i}. {q}")

In [ ]:
# Run retrieval for each test query
all_test_results = []

print("="*80)
print("RUNNING TEST QUERIES")
print("="*80)

for query_idx, query in enumerate(test_queries, 1):
    print(f"\n{'='*80}")
    print(f"Query {query_idx}: {query}")
    print(f"{'='*80}")
    
    # Retrieve
    results_df = retrieve_dense(
        query=query,
        model=model,
        index=index,
        corpus_df=df_corpus,
        top_k=TOP_K
    )
    
    # Store results
    results_df['test_query_index'] = query_idx
    results_df['test_query_text'] = query
    all_test_results.append(results_df)
    
    # Print results
    print(f"\nTop {TOP_K} results:")
    for _, row in results_df.iterrows():
        print(f"\n  [{row['rank']}] Score: {row['similarity_score']:.4f}")
        print(f"      Source: {row['source']}")
        print(f"      Category: {row['category']}")
        print(f"      Chunk: {row['chunk_text'][:150]}...")
    
    print()

In [ ]:
# Combine all test results
df_test_results = pd.concat(all_test_results, ignore_index=True)

print(f"✓ Completed all test queries")
print(f"  - Total results: {len(df_test_results)}")
print(f"  - Results per query: {len(df_test_results) // len(test_queries)}")
print(f"\nResult columns: {list(df_test_results.columns)}")

## 12. Save Embeddings, Index, and Results

In [ ]:
# Save embeddings as numpy
embeddings_path = f"{INDEX_OUTPUT_DIR}/kaggle_dense_embeddings.npy"
np.save(embeddings_path, embeddings)
print(f"✓ Saved embeddings: {embeddings_path}")
print(f"  - Size: {Path(embeddings_path).stat().st_size / (1024*1024):.2f} MB")

In [ ]:
# Save FAISS index
index_path = f"{INDEX_OUTPUT_DIR}/kaggle_faiss_index.index"
faiss.write_index(index, index_path)
print(f"✓ Saved FAISS index: {index_path}")
print(f"  - Size: {Path(index_path).stat().st_size / (1024*1024):.2f} MB")

In [ ]:
# Save retrieval row mapping (links corpus df to FAISS index)
row_mapping = pd.DataFrame({
    'faiss_index': range(len(df_corpus)),
    'corpus_row_index': df_corpus.index,
    'chunk_id': df_corpus['chunk_id'],
    'doc_id': df_corpus['doc_id']
})

mapping_path = f"{INDEX_OUTPUT_DIR}/kaggle_retrieval_row_mapping.csv"
row_mapping.to_csv(mapping_path, index=False)
print(f"✓ Saved row mapping: {mapping_path}")

In [ ]:
# Save test results as CSV
test_results_csv = f"{INDEX_OUTPUT_DIR}/kaggle_dense_retrieval_results.csv"
df_test_results.to_csv(test_results_csv, index=False)
print(f"✓ Saved test results (CSV): {test_results_csv}")
print(f"  - Size: {Path(test_results_csv).stat().st_size / 1024:.2f} KB")

In [ ]:
# Save test results as JSONL
test_results_jsonl = f"{INDEX_OUTPUT_DIR}/kaggle_dense_retrieval_results.jsonl"
with open(test_results_jsonl, 'w', encoding='utf-8') as f:
    for idx, row in df_test_results.iterrows():
        json_record = row.to_dict()
        # Convert NaN to None for JSON
        json_record = {
            k: (None if pd.isna(v) else v) for k, v in json_record.items()
        }
        f.write(json.dumps(json_record, ensure_ascii=False) + "\n")

print(f"✓ Saved test results (JSONL): {test_results_jsonl}")
print(f"  - Size: {Path(test_results_jsonl).stat().st_size / 1024:.2f} KB")

In [ ]:
# Save corpus embeddings metadata
metadata = {
    'dataset_variant': 'kaggle_only',
    'corpus_size': len(df_corpus),
    'embedding_model': EMBEDDING_MODEL_NAME,
    'embedding_dimension': EMBEDDING_DIMENSION,
    'embeddings_shape': embeddings.shape,
    'index_size': index.ntotal,
    'files': {
        'corpus_embeddings': 'kaggle_dense_embeddings.npy',
        'faiss_index': 'kaggle_faiss_index.index',
        'row_mapping': 'kaggle_retrieval_row_mapping.csv',
        'test_results_csv': 'kaggle_dense_retrieval_results.csv',
        'test_results_jsonl': 'kaggle_dense_retrieval_results.jsonl'
    }
}

metadata_path = f"{INDEX_OUTPUT_DIR}/kaggle_dense_retrieval_metadata.json"
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"✓ Saved metadata: {metadata_path}")

## 13. Summary and Next Steps

In [ ]:
print("\n" + "="*80)
print("DENSE RETRIEVAL BASELINE COMPLETE")
print("="*80)

print(f"\nCorpus Statistics:")
print(f"  • Corpus size: {len(df_corpus):,} chunks")
print(f"  • Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"  • Embedding dimension: {EMBEDDING_DIMENSION}")
print(f"  • Embeddings shape: {embeddings.shape}")
print(f"  • FAISS index size: {index.ntotal}")

print(f"\nTest Results:")
print(f"  • Test queries: {len(test_queries)}")
print(f"  • Top-K per query: {TOP_K}")
print(f"  • Total retrieval results: {len(df_test_results)}")

print(f"\nOutput Files (saved to {INDEX_OUTPUT_DIR}):")
output_files = [
    ("corpus_embeddings.npy", "Corpus embeddings (embeddings x dimension)"),
    ("faiss_index.bin", "FAISS vector index for cosine similarity search"),
    ("retrieval_row_mapping.csv", "Mapping from FAISS index to corpus dataframe"),
    ("dense_test_results.csv", "Retrieval results for 5 test queries"),
    ("dense_test_results.jsonl", "Same results in JSONL format"),
    ("dense_retrieval_metadata.json", "Metadata about embeddings, model, index"),
]

for fname, desc in output_files:
    path = f"{INDEX_OUTPUT_DIR}/{fname}"
    if Path(path).exists():
        size = Path(path).stat().st_size
        size_str = f"{size / (1024*1024):.2f} MB" if size > 1024*1024 else f"{size / 1024:.2f} KB"
        print(f"  ✓ {fname}")
        print(f"    {desc} ({size_str})")

In [ ]:
print(f"\nArchitecture Progress:")
print(f"  ✓ Step 1: Dataset preparation")
print(f"  ✓ Step 2: Retrieval corpus preparation")
print(f"  ✓ Step 3: Dense retrieval baseline (this notebook)")
print(f"  → Step 4: Optional reranking layer")
print(f"  → Step 5: LLM answer generation")
print(f"  → Step 6: End-to-end RAG evaluation")

print(f"\nNext Steps:")
print(f"  1. Review test retrieval results (kaggle_dense_retrieval_results.csv)")
print(f"  2. Implement BM25/keyword search in parallel notebook")
print(f"  3. Add reranking (cross-encoder) over top retrieval results")
print(f"  4. Integrate LLM for answer generation")
print(f"  5. Build formal benchmark for retrieval + RAG evaluation")
print(f"  6. Deploy as Gradio interface or API")

print(f"\nNotes:")
print(f"  • Embeddings are L2-normalized for cosine similarity search")
print(f"  • FAISS uses inner product for fast retrieval")
print(f"  • Model supports Turkish and 100+ languages")
print(f"  • This is the baseline dense retrieval stage only")
print(f"  • Future notebooks can add reranking and answer generation")